In [ ]:
import pandas as pd
import numpy as np
import torch
import torch.nn as nn
import torch.optim as optim
from sklearn.preprocessing import LabelEncoder, StandardScaler
from google.colab import drive
import os

drive.mount('/content/drive')

data_path = '/content/drive/MyDrive/F1_2026_Project/data/f1_historical_cleaned.csv'
df = pd.read_csv(data_path)



Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [ ]:
print(df.columns)
#used this to fix an error the nect code block was throwing

Index(['Year', 'Track', 'Driver', 'Team', 'Engine', 'Grid Position',
       'FinishPosition', 'Status'],
      dtype='object')


In [ ]:
le_track = LabelEncoder()
le_engine = LabelEncoder()

df['Track_Enc'] = le_track.fit_transform(df['Track'])
df['Engine_Enc'] = le_engine.fit_transform(df['Engine'])

X = df[['Grid Position', 'Track_Enc', 'Engine_Enc']].values.astype(np.float32)
y = df['FinishPosition'].values.reshape(-1 , 1).astype(np.float32)

scalar = StandardScaler()
X_scaled = scalar.fit_transform(X)

#standardizing target as well
y_scalar = StandardScaler()
y_scaled = y_scalar.fit_transform(y)
y_tensor = torch.from_numpy(y_scaled).float()

X_tensor = torch.from_numpy(X_scaled)
y_tensor = torch.from_numpy(y)

In [ ]:
# Check for NaNs in the raw data
print("--- Missing Value Count ---")
print(df.isna().sum())

# Check if targets (y) contain NaNs
print("\nAny NaNs in y_tensor?", torch.isnan(y_tensor).any().item())

# Check if inputs (X) contain NaNs
print("Any NaNs in X_tensor?", torch.isnan(X_tensor).any().item())

--- Missing Value Count ---
Year              0
Track             0
Driver            0
Team              0
Engine            0
Grid Position     1
FinishPosition    0
Status            0
Track_Enc         0
Engine_Enc        0
dtype: int64

Any NaNs in y_tensor? False
Any NaNs in X_tensor? True


In [ ]:

df['Grid Position'] = pd.to_numeric(df['Grid Position'], errors='coerce').fillna(20)


print("Remaining NaNs in Grid Position:", df['Grid Position'].isna().sum())


X = df[['Grid Position', 'Track_Enc', 'Engine_Enc']].values.astype('float32')
y = df['FinishPosition'].values.reshape(-1, 1).astype('float32')


scalar = StandardScaler()
X_scaled = scalar.fit_transform(X)

X_tensor = torch.from_numpy(X_scaled)
y_tensor = torch.from_numpy(y)

print("✅ X_tensor NaN check:", torch.isnan(X_tensor).any().item())

Remaining NaNs in Grid Position: 0
✅ X_tensor NaN check: False


In [ ]:
class F1Oracle(nn.Module):
  def __init__(self, input_dim):
    super(F1Oracle, self).__init__()
    self.layers = nn.Sequential(
        nn.Linear(input_dim ,64),
        nn.ReLU(),
        nn.Linear(64 , 32),
        nn.ReLU(),
        nn.Linear(32 , 1)
    )

  def forward(self,x):
    return self.layers(x)


In [ ]:

torch.autograd.set_detect_anomaly(True)

model = F1Oracle(X.shape[1])
optimizer = optim.Adam(model.parameters(), lr=0.001)
criterion = nn.MSELoss()

print("Training with stability checks...")
for epoch in range(501):
    optimizer.zero_grad()
    outputs = model(X_tensor)


    if torch.isnan(outputs).any():
        print(f"❌ NaN detected at epoch {epoch}! Stopping.")
        break

    loss = criterion(outputs, y_tensor)
    loss.backward()


    nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)

    optimizer.step()

    if epoch % 100 == 0:
        print(f"Epoch {epoch} | Loss: {loss.item():.4f}")

Training with stability checks...
Epoch 0 | Loss: 145.6951
Epoch 100 | Loss: 44.4004
Epoch 200 | Loss: 20.4028
Epoch 300 | Loss: 18.5406
Epoch 400 | Loss: 18.2260
Epoch 500 | Loss: 17.9567


In [ ]:
import os # Import os for os.makedirs

model = F1Oracle(X.shape[1])

criterion = nn.MSELoss()
optimizer = optim.Adam(model.parameters(), lr=0.01)

print("Starting Training")

for epoch in range(500):
  optimizer.zero_grad()
  predictions = model(X_tensor)
  loss = criterion(predictions, y_tensor)

  loss.backward()
  optimizer.step()

  if epoch%10 == 0:
    print(f"Epoch {epoch} | Loss: {loss.item():.4f}")

model_save_path = '/content/drive/MyDrive/F1_2026_Project/models/f1_v1.pth'
os.makedirs('/content/drive/MyDrive/F1_2026_Project/models' , exist_ok=True)
torch.save(model.state_dict(), model_save_path)
print("Model trained and saved")

Starting Training
Epoch 0 | Loss: 148.3760
Epoch 10 | Loss: 55.7592
Epoch 20 | Loss: 35.3776
Epoch 30 | Loss: 27.2239
Epoch 40 | Loss: 24.3785
Epoch 50 | Loss: 22.4966
Epoch 60 | Loss: 21.7438
Epoch 70 | Loss: 21.0141
Epoch 80 | Loss: 20.4845
Epoch 90 | Loss: 20.0300
Epoch 100 | Loss: 19.6879
Epoch 110 | Loss: 19.4320
Epoch 120 | Loss: 19.2301
Epoch 130 | Loss: 19.0882
Epoch 140 | Loss: 18.9890
Epoch 150 | Loss: 18.9201
Epoch 160 | Loss: 18.8658
Epoch 170 | Loss: 18.8114
Epoch 180 | Loss: 18.7724
Epoch 190 | Loss: 18.7341
Epoch 200 | Loss: 18.7106
Epoch 210 | Loss: 18.6879
Epoch 220 | Loss: 18.6664
Epoch 230 | Loss: 18.6477
Epoch 240 | Loss: 18.6266
Epoch 250 | Loss: 18.6040
Epoch 260 | Loss: 18.5819
Epoch 270 | Loss: 18.5642
Epoch 280 | Loss: 18.5469
Epoch 290 | Loss: 18.5287
Epoch 300 | Loss: 18.5131
Epoch 310 | Loss: 18.4993
Epoch 320 | Loss: 18.4882
Epoch 330 | Loss: 18.4757
Epoch 340 | Loss: 18.4631
Epoch 350 | Loss: 18.4523
Epoch 360 | Loss: 18.4414
Epoch 370 | Loss: 18.4302
Epoc

In [ ]:
import joblib

# Create a folder for your "serialized" objects
os.makedirs('/content/drive/MyDrive/F1_2026_Project/models/encoders', exist_ok=True)

# Save the track/engine encoders and the scaler
joblib.dump(le_track, '/content/drive/MyDrive/F1_2026_Project/models/encoders/le_track.joblib')
joblib.dump(le_engine, '/content/drive/MyDrive/F1_2026_Project/models/encoders/le_engine.joblib')
joblib.dump(scalar, '/content/drive/MyDrive/F1_2026_Project/models/encoders/scaler.joblib')

print("✅ Encoders saved!")

✅ Encoders saved!


In [ ]:
test_load = torch.load('/content/drive/MyDrive/F1_2026_Project/models/f1_v1.pth')
for key in test_load:
    if torch.isnan(test_load[key]).any():
        print(f"❌ Target weights are still poisoned in {key}!")

In [ ]:
import joblib

joblib.dump(scalar, '/content/drive/MyDrive/F1_2026_Project/models/scaler.joblib')

['/content/drive/MyDrive/F1_2026_Project/models/scaler.joblib']

In [ ]:
import joblib


joblib.dump(y_scalar, '/content/drive/MyDrive/F1_2026_Project/models/encoders/y_scaler.joblib')

print("✅ Target Scaler saved!")

✅ Target Scaler saved!
